# 08. Dual-stream CrossDiT K6: два потока Audio + Video

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** воспроизвести оба направления fusion, проверить gates и измерить межмодальную чувствительность/градиент

**Перед началом:** K6 07; Q/K/V и RoPE из 06. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/06_dit_and_flow_matching.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.2, Figures 1–2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/components/rope.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/rope.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','kandinsky-6-components'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '08_dual_stream_audio_video_crossdit'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Dual-stream — два набора tokens, не stereo audio

Здесь «два канала» означают **две модальности**. Video hidden width=1792, audio=896,
heads соответственно 28 и 14 при head_dim=64; у каждого потока 32 блока.
Два потока имеют свои text-refiner, timestep embeddings, self-attention и FFN.
Они обмениваются контекстом в каждом `FusedTransformerDecoderBlock`.

Основное чтение: [K6 §3.2 / Figures 1–2](https://arxiv.org/html/2610.05608v1#S3.SS2),
[страница статьи](https://huggingface.co/papers/2610.05608).
Сравнение с первичными работами: [DiT](https://arxiv.org/abs/2212.09748),
[Ovi: Twin Backbone Cross-Modal Fusion](https://arxiv.org/abs/2510.01284),
[LTX-2: Efficient Joint Audio-Visual Foundation Model](https://arxiv.org/abs/2601.03233).
У этих систем близкая идея blockwise fusion, но свои codec/text/normalization contracts.

### Направление определяется по query

$$
\Delta V = \mathrm{Attention}(V W_Q^v, A W_K^{a\to v}, A W_V^{a\to v}),\qquad
\Delta A = \mathrm{Attention}(A W_Q^a, V W_K^{v\to a}, V W_V^{v\to a}).
$$

Первая операция обновляет V, вторая A. В native коде `va_cross_attention` имеет video Q,
`av_cross_attention` — audio Q. Названия направлений в diagrams легко прочитать наоборот;
проверяйте destination по shape Q, а не по буквам имени.


In [ ]:
D=native()
v_from_a=D.MultiheadCrossAttention(48,8,32,engine='sdpa')
a_from_v=D.MultiheadCrossAttention(32,8,48,engine='sdpa')
v=torch.randn(1,8,48);a=torch.randn(1,6,32)
with torch.no_grad():dv=v_from_a(v,a);da=a_from_v(a,v)
assert dv.shape==v.shape and da.shape==a.shape
metrics['fusion_projection_shapes']={
    'video_Q':list(v_from_a.to_query.weight.shape),'audio_K_to_video':list(v_from_a.to_key.weight.shape),
    'audio_Q':list(a_from_v.to_query.weight.shape),'video_K_to_audio':list(a_from_v.to_key.weight.shape)}
print(json.dumps(metrics['fusion_projection_shapes'],indent=2))
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(11,4.3));ax.set_xlim(0,12);ax.set_ylim(0,5);ax.axis('off')
for y,label,color in [(3.4,'VIDEO: width 1792 / 28 heads','#d8eafa'),(1.2,'AUDIO: width 896 / 14 heads','#fbe3cc')]:
    ax.text(.15,y+.8,label,fontsize=10,weight='bold')
    for i,title in enumerate(['Self-attn','Text CA','AV fusion','FFN']):
        x=.3+2.9*i
        ax.add_patch(FancyBboxPatch((x,y),2.3,.65,boxstyle='round,pad=.1',facecolor=color,edgecolor='#555555'))
        ax.text(x+1.15,y+.32,title,ha='center',va='center')
        if i<3:ax.annotate('',xy=(x+2.8,y+.32),xytext=(x+2.3,y+.32),arrowprops={'arrowstyle':'->'})
ax.annotate('audio K/V -> video Q',xy=(7.25,3.35),xytext=(5.15,2.45),fontsize=9,
            arrowprops={'arrowstyle':'->','color':'#986b43'})
ax.annotate('video K/V -> audio Q',xy=(7.25,1.95),xytext=(8.3,2.45),fontsize=9,
            arrowprops={'arrowstyle':'->','color':'#426b98'})
ax.text(.15,.35,'Separate text refiners and time modulation; two cross outputs use saved pre-CA states.',fontsize=10)
fig.tight_layout();fig.savefig(figure_path(LAB,'dual_stream_architecture'));plt.show()


## 2. Точный порядок внутри FusedTransformerDecoderBlock

1. Video: modulated self-attention, затем вычисление text CA из `vis_pre_ca`.
2. Audio: modulated self-attention, text CA из `aud_pre_ca`, text residual update.
3. Video text residual update; подготовка modulated query states для двух cross-modal attentions.
4. Video queries читают **audio pre-text-CA normalized state**, audio queries — **video pre-text-CA state**.
   Оба cross outputs вычисляются до их residual application; это не последовательное V→A→V с новым output.
5. Cross residual gates, затем отдельные FFN двух потоков.

`ca_rope=True` добавляет позиции к Q/K fusion. `fix_modulation=True` выбирает timestep
соответствующего query-потока. `cross_gates=True` меняет ширины/источник gate vectors:
video-sized gate берётся из `av_modulation`, audio-sized — из `va_modulation`.
Следующий вывод показывает **полное оригинальное тело блока**, чтобы эти детали были проверяемыми.


In [ ]:
show_source('core/components/dit.py','FusedTransformerDecoderBlock')


## 3. Zero-init invariance, затем причинное отключение связей

Сначала проверим identity на только что созданном блоке: residual gates нулевые.
Затем явно заменим modulation weights малыми случайными значениями, чтобы связь стала наблюдаемой.
Это исследование функции, не синхронизации pretrained звука.

Для одного блока изменение A должно влиять на V при активной `va`-ветви;
при `va_gate_scale=0` это влияние исчезает. Аналогично проверяем обратное направление.
Отключение одной ветви в одном блоке не изолирует путь через другие блоки полной модели.


In [ ]:
model=tiny_model().eval();block=model.visual_transformer_blocks[0]
vr=model.visual_rope((2,2,2),[torch.arange(2)]*3).flatten(0,2)
ar=model.audio_rope(torch.arange(6))
v=torch.randn(1,8,48);a=torch.randn(1,6,32)
tv=torch.randn(1,1,16);ta=torch.randn(1,1,16)
text_v=torch.randn(1,5,48);text_a=torch.randn(1,5,32)
def run(vv,aa,va=1.,av=1.):
    return block(vv,aa,text_v,text_a,(tv,ta),vr,ar,
                 va_gate_scale=va,av_gate_scale=av)
with torch.no_grad():v0,a0=run(v,a)
identity=max((v0-v).abs().max().item(),(a0-a).abs().max().item());assert identity==0
activate_gates(block)
perturb_a=torch.randn_like(a);perturb_v=torch.randn_like(v)
with torch.no_grad():
    base_v,base_a=run(v,a)
    changed_v,_=run(v,a+perturb_a)
    _,changed_a=run(v+perturb_v,a)
    off_v,_=run(v,a,va=0);off_v2,_=run(v,a+perturb_a,va=0)
    _,off_a=run(v,a,av=0);_,off_a2=run(v+perturb_v,a,av=0)
rows={'A affects V: active':(changed_v-base_v).abs().max().item(),
      'A affects V: cut':(off_v2-off_v).abs().max().item(),
      'V affects A: active':(changed_a-base_a).abs().max().item(),
      'V affects A: cut':(off_a2-off_a).abs().max().item()}
assert rows['A affects V: active']>0 and rows['V affects A: active']>0
assert rows['A affects V: cut']==0 and rows['V affects A: cut']==0
metrics['zero_gate_identity_error']=identity;metrics['directional_perturbations']=rows
fig,ax=plt.subplots(figsize=(8,3));ax.barh(list(rows),list(rows.values()));ax.set_xlabel('Max output perturbation')
fig.tight_layout();fig.savefig(figure_path(LAB,'cross_modal_ablation'));plt.show();print(rows)


## 4. Gradient показывает путь обучения, не качество синхронизации

Пусть scalar loss зависит только от audio output. При открытом fusion он может дать gradient
на video input. Это объясняет, почему совместное обучение отличается от независимых T2V/T2A.
Но наличие gradient само по себе не доказывает lip-sync или голосовое сходство.


In [ ]:
vg=v.clone().requires_grad_()
_,audio_on=run(vg,a)
g_on=torch.autograd.grad(audio_on.square().mean(),vg)[0]
_,audio_off=run(vg,a,av=0)
g_off=torch.autograd.grad(audio_off.square().mean(),vg)[0]
metrics['audio_loss_gradient_to_video_on']=g_on.norm().item()
metrics['audio_loss_gradient_to_video_off']=g_off.norm().item()
assert g_on.norm()>0 and g_off.norm()==0
joint=tiny_model().eval();_,_,kw=tiny_inputs(joint)
with torch.no_grad():out=fresh_forward(joint,kw)
metrics['joint_native_output_shapes']=[list(x.shape) for x in out]
print('Joint shapes:',metrics['joint_native_output_shapes'],'gradient norms:',g_on.norm().item(),g_off.norm().item())


## 5. Joint probability не означает одинаковые clocks

Video latent positions и audio latent positions имеют разную частоту. Cross-attention матрица
прямоугольная Nv×Na; ей не нужен один audio token на один video token.
Два noise times могут быть разными: задаём `[t_v,t_a]`. Чистая известная модальность при t=0
может выступать условием для второй. Native audio-only forward при `x_video=None` пропускает fusion;
stock base-model π-Flow sampler требует оба latent tensors даже при частичном sampling.


## Самостоятельное исследование

1. Повторите ablation при `ca_rope=False`: измерьте sensitivity, не делайте вывода о lip-sync без pretrained eval.
2. Выведите shapes всех split vectors при `cross_gates=True`. Почему swapped gate имеет правильную width?
3. Переставьте audio tokens и их RoPE одновременно, затем только tokens: какие инвариантности вы ожидаете?
4. На бумаге сравните K6, Ovi и LTX-2: codec, text encoder, widths, fusion Q/KV, time conditioning.
5. Для voice-FiLM гипотезы отметьте insertion point и проверьте путь gradient к адаптеру при frozen backbone.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете нарисовать два полных потока, подписать Q/K/V и timestep каждого attention,
воспроизвести directional cut и объяснить, почему toy sensitivity не равна AV synchronization metric.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 07. От DiT к CrossDiT: как denoiser устроен изнутри](07_dit_and_text_crossdit.ipynb) · [09. Joint Flow Matching: path, velocity и учебное обучение →](09_flow_matching_training.ipynb)
